# Stage D / NB 18 — calibration, ROC and operating points

Protocol reference: **R1.3** (statistics), **E8d** (decision-threshold tuning), metrics §7.3.

## Why this is a separate notebook from NB 17

NB 17 answers "is arm A better than arm B". This one answers "at what threshold would you
actually use it, and does its confidence mean anything" — and those are the questions a
clinical reviewer asks second.

The previous submission reported precision and F1 without confusion counts, which hid a
specificity collapse. Every operating point here carries its full confusion matrix.

## The rule that makes an operating point honest

**A threshold is selected on inner-validation rows and applied to test rows.** Choosing the
threshold that maximises Youden's J *on the test set* and then reporting the sensitivity and
specificity it achieves there is circular — it reports the best of ~n thresholds as if it were
one. The gate below fails if any arm's threshold was selected on data it is then scored on.

Membership comes from NB 13's fold-relative `inner_fold_k` columns. "Inner validation" is a
relation between an image and a fold, not a property of the image.

## Endpoint and denominator contract

NB 18 inherits NB 17's endpoint-specific usability decisions. NB 14 intentionally writes
separate mRALE and COVID rows; only the COVID-bearing row contributes to this notebook, and
duplicate COVID rows for an `(arm, image_key)` are blocking. Invalid or out-of-range scores
are retained as the protocol-specified uninformative value 0.5, never selectively dropped.

## Two operating points, both pre-declared

| point | rule | why |
| --- | --- | --- |
| Youden-J | maximise sensitivity + specificity − 1 on inner validation | the conventional balanced choice |
| fixed sensitivity 0.90 | highest threshold reaching 0.90 sensitivity on inner validation | screening-style use, where missing a case is the expensive error |

## Calibration

Brier score and expected calibration error, plus a reliability diagram per arm. A model can
have a respectable AUROC and be badly calibrated — the ranking is fine while the numbers it
reports are not probabilities. Given that Stage B found detection AUROC barely above chance,
the calibration panel is likely to be the more informative one.

## Outputs (under `stage_D/nb18_calibration/`)
`roc_pr_curves.csv`, `calibration.csv`, `operating_points.csv`,
`operating_point_comparisons.csv`, `inner_score_sources.csv`,
`figures/fig3_roc_pr.pdf|svg`, `figures/fig4_calibration.pdf|svg`, `gate_nb18.json`.


## 1. Imports and the locked NB 17 artifacts

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Metric definitions are shared with Stage B/C. Every number in the manuscript must come from
# the same code that produced the arm tables, or the tables and the statistics disagree.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

# Stage D's own statistics module: bootstrap indices drawn once, DeLong, McNemar, Holm, TOST,
# and the rule that a p-value cannot exist without its metadata.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_D",
                   Path.cwd().parent.parent / "notebooks" / "stage_D"]:
    if (_candidate / "stage_d_stats.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("stage_d_stats.py not found; it must sit beside these notebooks.")
import stage_d_stats as sd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_D_DIR = STAGE_ROOT / "stage_D"
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
N_BOOTSTRAP = sd.BOOTSTRAP_REPLICATES
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; guard section boundaries
SESSION_DEADLINE = sd.make_session_deadline(MAX_SESSION_HOURS)

print("Stage D output:", STAGE_D_DIR)
print(f"Bootstrap: {N_BOOTSTRAP} patient-level replicates, seed {sd.BOOTSTRAP_SEED}")
print(f"Soft stop: {MAX_SESSION_HOURS:.1f} h after setup; checks occur between sections")

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 3")

def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level.")
    frame = pd.read_csv(path)
    frame["image_key"] = "MIDRC::" + frame["filename"].astype(str)
    return frame


# Every internal out-of-fold prediction file Stage B and Stage C can produce.
# (label, family, directory, filename). `family` drives the multiplicity families of 8.6.
INTERNAL_SOURCES = [
    ("A5_cxformer",        "E0",  STAGE_B_DIR / "nb05_frozen_encoder",     "predictions_frozen.jsonl"),
    ("E0g_conventional",   "E0",  STAGE_B_DIR / "nb06_conventional",       "predictions_conventional.jsonl"),
    ("E0_zeroshot",        "E0",  STAGE_B_DIR / "nb07_zeroshot",           "predictions_zeroshot.jsonl"),
    ("A6_biomedclip",      "E0",  STAGE_B_DIR / "nb08_biomedclip_entity",  "predictions_entity_probe.jsonl"),
    ("A2_medgemma_lora",   "E0",  STAGE_B_DIR / "nb09_medgemma_lora",      "predictions_medgemma_lora.jsonl"),
    ("A3_qwen_lora",       "E0",  STAGE_B_DIR / "nb10_qwen_lora",          "predictions_qwen_lora.jsonl"),
    ("A4_nvreason",        "E0",  STAGE_B_DIR / "nb11_nvreason",           "predictions_nvreason.jsonl"),
    ("E4_anatomy",         "E4",  STAGE_B_DIR / "nb12_anatomy_aware",      "anatomy_aware_predictions.jsonl"),
    ("E7_fusion",          "E7",  STAGE_C_DIR / "nb14_fusion",             "fusion_predictions.jsonl"),
]
EXTERNAL_SOURCES = [
    ("A5_cxformer",      STAGE_B_DIR / "nb05_frozen_encoder",    "external_predictions.jsonl"),
    ("E0g_conventional", STAGE_B_DIR / "nb06_conventional",      "external_predictions.jsonl"),
    ("A6_biomedclip",    STAGE_B_DIR / "nb08_biomedclip_entity", "external_predictions.jsonl"),
    ("A2_medgemma_lora", STAGE_B_DIR / "nb09_medgemma_lora",     "external_predictions.jsonl"),
    ("A3_qwen_lora",     STAGE_B_DIR / "nb10_qwen_lora",         "external_predictions.jsonl"),
]


def discover_arms(log=print):
    """
    Build the arm table from whatever Stage B and Stage C actually produced.

    Absence is recorded, never inferred: a notebook that has not run is a different thing from
    an arm that produced nothing, and the two have different remedies.
    """
    rows, availability = [], []
    for label, family, directory, filename in INTERNAL_SOURCES:
        path = directory / filename
        if not path.is_file():
            availability.append({"source": label, "family": family, "path": str(path),
                                 "status": "MISSING", "n_rows": 0, "n_arms": 0,
                                 "reason": "prediction file not found; notebook not yet run"})
            continue
        found = cm.read_jsonl(path)
        arms = sorted({str(r.get("arm", label)) for r in found})
        availability.append({"source": label, "family": family, "path": str(path),
                             "status": "OK", "n_rows": len(found), "n_arms": len(arms),
                             "reason": ""})
        for row in found:
            row["_source"] = label
            row["_family"] = family
            row["arm"] = str(row.get("arm", label))
            rows.append(row)

    # Stage C reasoner arms live in one file per roster.
    reasoner_dir = STAGE_C_DIR / "nb15_reasoner"
    reasoner_files = sorted(reasoner_dir.glob("reasoner_predictions_*.jsonl"))
    if reasoner_files:
        for path in reasoner_files:
            arm = path.stem.replace("reasoner_predictions_", "")
            found = cm.read_jsonl(path)
            family = "E1" if arm.startswith("E1") else "E7"
            availability.append({"source": f"reasoner/{arm}", "family": family,
                                 "path": str(path), "status": "OK", "n_rows": len(found),
                                 "n_arms": 1, "reason": ""})
            for row in found:
                row["_source"] = "reasoner"
                row["_family"] = family
                row["arm"] = arm
                rows.append(row)
    else:
        availability.append({"source": "reasoner", "family": "E1",
                             "path": str(reasoner_dir / "reasoner_predictions_*.jsonl"),
                             "status": "MISSING", "n_rows": 0, "n_arms": 0,
                             "reason": "NB 15 has not produced any roster predictions"})
    for entry in availability:
        log(f"  [{entry['status']:<7}] {entry['source']:<24} rows={entry['n_rows']:<7} "
            f"arms={entry['n_arms']}")
    return rows, pd.DataFrame(availability)

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 4")

NB18_DIR = STAGE_D_DIR / "nb18_calibration"
FIGURE_DIR = NB18_DIR / "figures"
for directory in (NB18_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
NB17_DIR = STAGE_D_DIR / "nb17_statistics"      # READ ONLY
NB13_DIR = STAGE_C_DIR / "nb13_registry"        # READ ONLY
NB14_DIR = STAGE_C_DIR / "nb14_fusion"          # READ ONLY
NB15_DIR = STAGE_C_DIR / "nb15_reasoner"        # READ ONLY


def require_upstream_gate(directory, notebook_number):
    candidates = [directory / f"gate_nb{notebook_number:02d}.json",
                  directory / f"gate_nb{notebook_number}.json"]
    path = next((candidate for candidate in candidates if candidate.is_file()), candidates[0])
    if not path.is_file():
        raise FileNotFoundError(
            f"Required upstream gate is missing: {path}. Run NB {notebook_number} to "
            "completion before NB 18; partial artifacts must not enter calibration.")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not bool(payload.get("passed", False)):
        raise RuntimeError(
            f"NB {notebook_number} did not pass its gate: {payload.get('failures', [])}")
    return {"path": str(path), "passed": True}


UPSTREAM_GATES = {
    "NB13": require_upstream_gate(NB13_DIR, 13),
    "NB17": require_upstream_gate(NB17_DIR, 17),
}

metrics_path = NB17_DIR / "all_metrics_with_ci.csv"
usability_path = NB17_DIR / "endpoint_usability.csv"
config_path = NB17_DIR / "run_config.json"
nb14_target_path = NB14_DIR / "e7f_target.json"
for required in [metrics_path, usability_path, config_path, nb14_target_path]:
    if not required.is_file():
        raise FileNotFoundError(
            f"{required} not found. NB 18 consumes the completed NB 17 endpoint contract; "
            "rerun NB 17 rather than reconstructing or guessing it here.")

locked_metrics = pd.read_csv(metrics_path)
endpoint_usability = pd.read_csv(usability_path)
nb17_config = json.loads(config_path.read_text(encoding="utf-8"))
nb14_target = json.loads(nb14_target_path.read_text(encoding="utf-8"))
REFERENCE_ARM = str(nb17_config["reference_arm"])
MRALE_STACKING_ARM = str(nb17_config["stacking_arm"])
# NB 14 locks endpoint-specific stacking arms. NB 18 is a COVID-score notebook, so it
# must not use the mRALE regressor when the locked COVID stack is logistic regression.
COVID_STACKING_ARM = str(
    nb17_config.get("covid_stacking_arm")
    or nb14_target.get("best_covid_stacking_arm") or "").strip()
if not COVID_STACKING_ARM:
    raise RuntimeError(
        "NB 14 did not lock best_covid_stacking_arm; rerun NB 14 before NB 18.")
STACKING_ARM = COVID_STACKING_ARM  # endpoint-specific alias used below

if endpoint_usability["arm"].astype(str).duplicated().any():
    duplicates = endpoint_usability.loc[
        endpoint_usability["arm"].astype(str).duplicated(keep=False), "arm"].tolist()
    raise RuntimeError(f"NB 17 endpoint_usability.csv has duplicate arms: {duplicates[:5]}")


def optional_csv_bool(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return True
    if text in {"false", "0", "no"}:
        return False
    raise RuntimeError(f"Unrecognised endpoint-usability Boolean: {value!r}")


score_usable_by_arm = {
    str(row["arm"]): optional_csv_bool(row.get("score_usable"))
    for row in endpoint_usability.to_dict("records")
}
declared_covid_rows_by_arm = {
    str(row["arm"]): (None if pd.isna(row.get("n_covid_rows"))
                      else int(row["n_covid_rows"]))
    for row in endpoint_usability.to_dict("records")
}
print(f"NB 17 locked {len(locked_metrics)} arms; reference arm {REFERENCE_ARM}")
print(f"NB 14 stacking arms: mRALE={MRALE_STACKING_ARM}; "
      f"COVID={COVID_STACKING_ARM} (used in NB 18)")

# Curves are drawn for the arms a reader will actually look at: the reference, the best
# baseline, and any fusion arm that beat it. Drawing dozens of ROC curves communicates nothing.
FIXED_SENSITIVITY = 0.90
N_THRESHOLD_GRID = 200
N_CALIBRATION_BINS = 10
# The all-agent COVID denominator is small (~95 images), and each 10% fold-relative
# development split can therefore contain fewer than 20 cases. Five two-class rows is
# the execution floor; sets below 20 remain explicitly flagged as unstable.
MIN_INNER_SELECTION_ROWS = 5
RECOMMENDED_INNER_SELECTION_ROWS = 20
MAX_CURVE_ARMS = 6
N_BAND_REPLICATES = N_BOOTSTRAP  # all 2,000 shared NB 17 patient-bootstrap replicates

print(f"Operating points: Youden-J and fixed sensitivity {FIXED_SENSITIVITY:.2f}, both "
      "selected on owner-produced inner validation")
print(f"ROC/PR bands reuse all {N_BAND_REPLICATES} shared patient bootstrap replicates")


## 2. Rebuild endpoint-specific scored rows and recover inner-validation membership

The predictions are re-read rather than recomputed. NB 14's separate mRALE and COVID rows are
merged by endpoint exactly as in NB 17: an mRALE-only row cannot become an uninformative COVID
observation. Each arm's COVID denominator and usability flag are checked against NB 17 before
any curve is calculated.

Every AUROC printed below is also checked against the value NB 17 locked. A disagreement means
the two notebooks are looking at different rows, which is worth failing on rather than
discovering in a figure caption.


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 6")

folds = load_folds()
patient_of = dict(zip(folds["image_key"].astype(str), folds["group_id"].astype(str)))
fold_of = dict(zip(folds["image_key"].astype(str), folds["fold"].astype(int)))
truth_covid = dict(zip(folds["image_key"].astype(str), folds["covid_positive"].astype(str)))


def endpoint_kind(row):
    task = str(row.get("task") or "").strip().lower()
    if task == "mrale_prediction":
        return "mrale"
    if task == "covid_classification":
        return "covid"
    return "multitask"


def bounded_covid_score(value):
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return False
    return math.isfinite(numeric) and 0.0 <= numeric <= 1.0


raw_rows, availability = discover_arms(log=lambda *a: None)
source_of_arm = {}
scored = defaultdict(list)
seen_covid_rows = set()
unknown_internal_keys = set()
for row in raw_rows:
    key = str(row.get("image_key", ""))
    if key not in truth_covid:
        # External prediction rows can coexist in source files. NB 19 owns them; an unknown
        # MIDRC-like row, however, indicates a stale internal cohort and is blocking.
        if key.startswith("MIDRC::"):
            unknown_internal_keys.add(key)
        continue
    if truth_covid[key] not in {"Yes", "No"} or endpoint_kind(row) == "mrale":
        continue
    arm = str(row["arm"])
    identity = (arm, key)
    if identity in seen_covid_rows:
        raise RuntimeError(
            f"Duplicate COVID-bearing prediction row for arm={arm}, image_key={key}. "
            "NB 14's mRALE/COVID rows must be separated by task, while repeated rows for the "
            "same endpoint must be repaired upstream.")
    seen_covid_rows.add(identity)
    source_of_arm.setdefault(arm, str(row.get("_source", "")))
    raw_score = row.get("covid_score")
    source_valid = bool(row.get("valid", False))
    score_present = source_valid and bounded_covid_score(raw_score)
    decision = row.get("covid_pred") if source_valid else None
    if decision not in {"Yes", "No"}:
        decision = None
    scored[arm].append({
        "image_key": key, "patient": patient_of[key], "fold": fold_of[key],
        "truth": 1 if truth_covid[key] == "Yes" else 0,
        # Protocol 7.4: invalid/missing/out-of-range is uninformative at 0.5, never dropped.
        "score": float(raw_score) if score_present else float(cm.INVALID_COVID_SCORE),
        "score_present": bool(score_present), "score_source_valid": source_valid,
        "decision": decision,
    })

if unknown_internal_keys:
    raise RuntimeError(
        f"Prediction artifacts contain {len(unknown_internal_keys)} MIDRC key(s) outside NB 02 "
        f"folds, e.g. {sorted(unknown_internal_keys)[:5]}. Stale cohort?")

denominator_issues = []
for arm, rows in sorted(scored.items()):
    keys = [row["image_key"] for row in rows]
    if len(keys) != len(set(keys)):
        denominator_issues.append(f"{arm}: duplicate COVID image keys after consolidation")
    declared = declared_covid_rows_by_arm.get(arm)
    if declared is None:
        denominator_issues.append(f"{arm}: absent from NB 17 endpoint_usability.csv")
    elif len(rows) != declared:
        denominator_issues.append(
            f"{arm}: NB 18 has {len(rows)} COVID rows but NB 17 locked {declared}")
# NB 17 also inventories NB 16's E6 exploratory sensitivity arms. NB 18 deliberately
# does not ingest those partial, fold-0-only experiments: they are not candidates for
# confirmatory calibration or operating-point selection. Keep the reverse denominator
# check strict for every non-exploratory arm, while excluding E6 by its declared family
# (with a prefix fallback for endpoint contracts written by an older NB 17).
usability_family = {str(row["arm"]): str(row.get("family", "")).upper()
                    for row in endpoint_usability.to_dict("records")}
confirmatory_declared_arms = {
    arm for arm in declared_covid_rows_by_arm
    if usability_family.get(arm) != "EXPLORATORY" and not arm.startswith("E6")}
for arm, declared in sorted(declared_covid_rows_by_arm.items()):
    if declared and arm in confirmatory_declared_arms and arm not in scored:
        denominator_issues.append(
            f"{arm}: NB 17 locked {declared} COVID rows but NB 18 discovered none")

# ---- Inner-validation membership, from NB 13's fold-relative columns ---------------------
registry = None
for candidate in [NB13_DIR / "agent_registry.parquet", NB13_DIR / "agent_registry.csv"]:
    if candidate.is_file():
        try:
            registry = (pd.read_parquet(candidate) if candidate.suffix == ".parquet"
                        else pd.read_csv(candidate))
            break
        except Exception:
            continue
if registry is None:
    raise FileNotFoundError(
        f"No agent registry under {NB13_DIR}. NB 13 defines which images are fold k's inner "
        "validation. Run NB 13 before this notebook.")

INNER_FOLD_COLUMNS = [f"inner_fold_{k}" for k in range(N_FOLDS)]
missing = [column for column in INNER_FOLD_COLUMNS if column not in registry.columns]
if missing:
    raise RuntimeError(
        f"The registry lacks {missing}. Re-run NB 13: inner-validation membership is a relation "
        "between an image and a fold, and a single split column cannot express it.")
membership = registry.drop_duplicates("image_key").set_index("image_key")[INNER_FOLD_COLUMNS]
inner_for_fold = {k: set(membership.index[membership[f"inner_fold_{k}"] == 1].astype(str))
                  for k in range(N_FOLDS)}
print("Inner-validation images available for threshold selection:")
for k in range(N_FOLDS):
    print(f"  fold {k}: {len(inner_for_fold[k]):,}")

# Fold-specific inner-validation scores. Invalid individual scores are retained as 0.5.
# An aggregated file is accepted only when every row declares which fold model produced it.
inner_scored = defaultdict(lambda: defaultdict(list))
inner_score_status = []
inner_duplicate_issues = []
INNER_SCORE_DIRS = [
    ("A2_medgemma_lora_final", STAGE_B_DIR / "nb09_medgemma_lora"),
    ("A3_qwen_lora_final", STAGE_B_DIR / "nb10_qwen_lora"),
    (STACKING_ARM, NB14_DIR),
    (REFERENCE_ARM, NB15_DIR),
]
for owner_arm, directory in INNER_SCORE_DIRS:
    for fold in range(N_FOLDS):
        path = directory / "folds" / f"fold_{fold}" / "inner_validation_score_records.jsonl"
        if not path.is_file():
            candidates = sorted(directory.glob("**/*inner*validation*score*.jsonl"))
            alternatives = [candidate for candidate in candidates if f"fold_{fold}" in str(candidate)]
            if len(alternatives) == 1:
                path = alternatives[0]
            elif len(candidates) == 1:
                path = candidates[0]
        rows_here = cm.read_jsonl(path) if path.is_file() else []
        accepted_by_key = {}
        for item in rows_here:
            # NB 14 writes every COVID fusion arm into the same per-fold file. Select
            # only the owner arm requested by this pass; otherwise rows for the other
            # fusion arms look like duplicate scores for the same image.
            item_arm = str(item.get("arm", owner_arm))
            if item_arm != owner_arm:
                continue
            if endpoint_kind(item) == "mrale":
                continue
            key = str(item.get("image_key", ""))
            raw_fold = item.get("fold", item.get("held_out_fold"))
            if raw_fold is None:
                # A per-fold path identifies the owner fold; an aggregate file must say.
                if f"fold_{fold}" not in str(path):
                    continue
                row_fold = fold
            else:
                try:
                    row_fold = int(raw_fold)
                except (TypeError, ValueError):
                    continue
            if key not in truth_covid or key not in inner_for_fold[fold] or row_fold != fold:
                continue
            if key in accepted_by_key:
                inner_duplicate_issues.append((owner_arm, fold, key, str(path)))
                continue
            raw_score = item.get("covid_score")
            declared_valid = item.get("valid")
            source_valid = (True if declared_valid is None else bool(declared_valid))
            score_present = source_valid and bounded_covid_score(raw_score)
            accepted_by_key[key] = {
                "image_key": key, "patient": patient_of[key], "fold": fold,
                "truth": 1 if truth_covid[key] == "Yes" else 0,
                "score": float(raw_score) if score_present else float(cm.INVALID_COVID_SCORE),
                "score_present": bool(score_present), "decision": item.get("covid_pred")}
        inner_scored[owner_arm][fold] = list(accepted_by_key.values())
        n_accepted = len(accepted_by_key)
        coverage = (float(np.mean([row["score_present"] for row in accepted_by_key.values()]))
                    if n_accepted else 0.0)
        inner_score_status.append({
            "arm": owner_arm, "fold": fold, "path": str(path), "n": n_accepted,
            "n_membership": len(inner_for_fold[fold]), "score_coverage": coverage})

# A genuinely frozen zero-shot arm has no fold-specific training state; its one prediction
# can be reused for membership-defined inner validation. No trained/fused/reasoning arm gets
# this exemption.
for arm, rows in scored.items():
    if source_of_arm.get(arm) != "E0_zeroshot":
        continue
    by_key = {row["image_key"]: row for row in rows}
    for fold in range(N_FOLDS):
        inner_scored[arm][fold] = [by_key[key] for key in inner_for_fold[fold] if key in by_key]
        selected = inner_scored[arm][fold]
        inner_score_status.append({
            "arm": arm, "fold": fold, "path": "fold-invariant zero-shot OOF score",
            "n": len(selected), "n_membership": len(inner_for_fold[fold]),
            "score_coverage": (float(np.mean([row["score_present"] for row in selected]))
                               if selected else 0.0)})
pd.DataFrame(inner_score_status).to_csv(NB18_DIR / "inner_score_sources.csv", index=False)

# Protocol 7.4 substitutes 0.5 for an INDIVIDUAL invalid score. An endpoint quarantined by
# NB 17 or an arm with almost no scores cannot enter discrimination/calibration.
MINIMUM_SCORE_COVERAGE = 0.50
score_coverage = {arm: (float(np.mean([row["score_present"] for row in rows])) if rows else 0.0)
                  for arm, rows in scored.items()}
ARMS_WITH_SCORES = sorted(
    arm for arm, rows in scored.items()
    if len(rows) >= 30 and len({row["truth"] for row in rows}) == 2
    and score_coverage[arm] >= MINIMUM_SCORE_COVERAGE
    and score_usable_by_arm.get(arm) is not False)
no_score = sorted(arm for arm, rows in scored.items()
                  if rows and (score_coverage[arm] < MINIMUM_SCORE_COVERAGE
                               or score_usable_by_arm.get(arm) is False))
print(f"\nArms with a usable continuous score: {len(ARMS_WITH_SCORES)} of {len(scored)}")
if no_score:
    print(f"  Arms quarantined by NB 17 or below {MINIMUM_SCORE_COVERAGE:.0%} score coverage, "
          f"EXCLUDED from ROC, PR and calibration: {no_score}")
    for arm in no_score[:5]:
        print(f"    {arm}: coverage {score_coverage[arm]:.3f}, "
              f"NB17 score_usable={score_usable_by_arm.get(arm)}")

# Cross-check against NB 17 rather than trusting either notebook.
disagreements = []
for arm in ARMS_WITH_SCORES:
    rows = scored[arm]
    here = sd.weighted_auroc(np.array([row["truth"] for row in rows], dtype=float),
                             np.array([row["score"] for row in rows], dtype=float),
                             np.ones(len(rows)))
    locked = locked_metrics.loc[locked_metrics["arm"] == arm, "covid_auroc"]
    if len(locked) and math.isfinite(float(locked.iloc[0])) and math.isfinite(here):
        if abs(float(locked.iloc[0]) - here) > 0.005:
            disagreements.append((arm, float(locked.iloc[0]), here))
print(f"\nAUROC cross-check against NB 17: {len(disagreements)} disagreement(s)")
for arm, locked_value, here in disagreements[:5]:
    print(f"  {arm}: NB 17 {locked_value:.4f} vs here {here:.4f}")


## 3. ROC and PR curves, with patient-level bootstrap bands

The bands reuse NB 17's cached bootstrap indices, so a band drawn here and an interval quoted
there describe the same resampling. Drawing fresh indices would produce a figure whose
uncertainty subtly disagrees with the table beside it.

Bands use the same 2,000 cached patient-bootstrap replicates as NB 17. This keeps the visual
uncertainty and the tabulated intervals on one locked resampling contract.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 8")

ALL_PATIENTS = sorted(set(patient_of.values()))
bootstrap = sd.PatientBootstrap(
    ALL_PATIENTS, n_replicates=N_BOOTSTRAP, seed=sd.BOOTSTRAP_SEED,
    cache_path=NB17_DIR / "bootstrap_indices.npz", read_only=True,
    expected_fingerprint=nb17_config.get("bootstrap_fingerprint"))
print(f"Bootstrap loaded read-only from NB 17 (fingerprint {bootstrap.fingerprint})")

GRID = np.linspace(0.0, 1.0, N_THRESHOLD_GRID + 1)
FPR_GRID = np.linspace(0.0, 1.0, N_THRESHOLD_GRID + 1)
RECALL_GRID = np.linspace(0.0, 1.0, N_THRESHOLD_GRID + 1)


def roc_pr_points(truth, score, weights=None):
    truth = np.asarray(truth, dtype=float)
    score = np.asarray(score, dtype=float)
    weights = np.ones_like(truth) if weights is None else np.asarray(weights, dtype=float)
    positives = (weights * truth).sum()
    negatives = (weights * (1 - truth)).sum()
    if positives <= 0 or negatives <= 0:
        return None
    tpr, fpr, precision = [], [], []
    for threshold in GRID:
        predicted = score >= threshold
        tp = (weights * predicted * truth).sum()
        fp = (weights * predicted * (1 - truth)).sum()
        tpr.append(tp / positives)
        fpr.append(fp / negatives)
        precision.append(tp / (tp + fp) if (tp + fp) > 0 else 1.0)
    return np.array(fpr), np.array(tpr), np.array(precision)


def interpolate_upper_envelope(x, y, grid):
    """Interpolate after resolving duplicate x values by the maximum y."""
    frame = pd.DataFrame({"x": np.asarray(x, dtype=float),
                          "y": np.asarray(y, dtype=float)})
    frame = frame.groupby("x", as_index=False)["y"].max().sort_values("x")
    return np.interp(grid, frame["x"].to_numpy(), frame["y"].to_numpy())


curve_rows = []
# Rank arms by locked AUROC so the figure shows the ones a reader would ask about.
ranked = (locked_metrics[locked_metrics["arm"].isin(ARMS_WITH_SCORES)]
          .sort_values("covid_auroc", ascending=False)["arm"].tolist())
CURVE_ARMS = ([REFERENCE_ARM] if REFERENCE_ARM in ARMS_WITH_SCORES else []) + \
             [a for a in ranked if a != REFERENCE_ARM][:MAX_CURVE_ARMS - 1]
print(f"Curves drawn for: {CURVE_ARMS}")

curve_data = {}
for arm in CURVE_ARMS:
    rows = scored[arm]
    truth = np.array([r["truth"] for r in rows], dtype=float)
    score = np.array([r["score"] for r in rows], dtype=float)
    positions = bootstrap.patient_positions([r["patient"] for r in rows])
    point = roc_pr_points(truth, score)
    if point is None:
        continue
    fpr, tpr, precision = point
    tpr_on_fpr = interpolate_upper_envelope(fpr, tpr, FPR_GRID)
    precision_on_recall = interpolate_upper_envelope(tpr, precision, RECALL_GRID)

    tpr_draws = np.full((N_BAND_REPLICATES, len(GRID)), np.nan)
    precision_draws = np.full((N_BAND_REPLICATES, len(GRID)), np.nan)
    for b in range(N_BAND_REPLICATES):
        weights = bootstrap.replicate_weights(positions, b)
        replicate = roc_pr_points(truth, score, weights)
        if replicate is None:
            continue
        tpr_draws[b] = interpolate_upper_envelope(replicate[0], replicate[1], FPR_GRID)
        precision_draws[b] = interpolate_upper_envelope(
            replicate[1], replicate[2], RECALL_GRID)
    tpr_low, tpr_high = (np.nanpercentile(tpr_draws, 2.5, axis=0),
                         np.nanpercentile(tpr_draws, 97.5, axis=0))
    precision_low, precision_high = (np.nanpercentile(precision_draws, 2.5, axis=0),
                                     np.nanpercentile(precision_draws, 97.5, axis=0))
    curve_data[arm] = {"fpr": FPR_GRID, "tpr": tpr_on_fpr, "recall": RECALL_GRID,
                       "precision": precision_on_recall,
                       "tpr_low": tpr_low, "tpr_high": tpr_high,
                       "precision_low": precision_low,
                       "precision_high": precision_high,
                       "auroc": float(locked_metrics.loc[locked_metrics["arm"] == arm,
                                                         "covid_auroc"].iloc[0])}
    for index in range(len(FPR_GRID)):
        curve_rows.append({"arm": arm, "grid_index": index,
                           "fpr": round(float(FPR_GRID[index]), 6),
                           "tpr": round(float(tpr_on_fpr[index]), 6),
                           "recall": round(float(RECALL_GRID[index]), 6),
                           "precision": round(float(precision_on_recall[index]), 6),
                           "precision_ci_low": round(float(precision_low[index]), 6),
                           "precision_ci_high": round(float(precision_high[index]), 6),
                           "tpr_ci_low": round(float(tpr_low[index]), 6),
                           "tpr_ci_high": round(float(tpr_high[index]), 6)})

roc_pr = pd.DataFrame(curve_rows, columns=["arm", "grid_index", "fpr", "tpr",
                                                "recall", "precision",
                                                "precision_ci_low", "precision_ci_high",
                                                "tpr_ci_low", "tpr_ci_high"])
roc_pr.to_csv(NB18_DIR / "roc_pr_curves.csv", index=False)
print(f"roc_pr_curves.csv: {len(roc_pr):,} rows over {len(curve_data)} arms")

## 4. Operating points selected only on inner validation

Each fold model receives its own threshold, selected from owner-produced inner-validation
scores and applied only to that fold's outer-test images. Pooled rows aggregate those five
held-out confusion matrices; their reported threshold is descriptive, not a sixth threshold
applied to the pooled test set.

For the locked reasoner-versus-stacking McNemar comparisons, NB 14's independently locked
**COVID stacking arm** is used—not its potentially different mRALE regressor. The COVID
stacking endpoint defines the
paired image set. The reference arm must cover that complete set. Silent post-hoc intersection
of whichever patients happen to remain is forbidden.


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 10")

def choose_threshold(rows, rule):
    """Pick a threshold on `rows`. Returns (threshold, achieved) or (None, reason)."""
    truth = np.array([row["truth"] for row in rows], dtype=float)
    score = np.array([row["score"] for row in rows], dtype=float)
    if truth.sum() == 0 or (1 - truth).sum() == 0:
        return None, "selection rows contain a single class"
    best, best_value = None, -np.inf
    for threshold in GRID:
        predicted = score >= threshold
        sensitivity = float((predicted * truth).sum() / truth.sum())
        specificity = float(((~predicted) * (1 - truth)).sum() / (1 - truth).sum())
        if rule == "youden":
            value = sensitivity + specificity - 1
        elif rule == "fixed_sensitivity":
            value = specificity if sensitivity >= FIXED_SENSITIVITY else -np.inf
        else:
            raise ValueError(rule)
        if value > best_value or (value == best_value and (best is None or threshold > best)):
            best, best_value = float(threshold), value
    if not math.isfinite(best_value):
        return None, f"no threshold reaches sensitivity {FIXED_SENSITIVITY}"
    return best, "ok"


def apply_threshold(rows, threshold):
    truth = np.array([row["truth"] for row in rows], dtype=float)
    predicted = np.array([row["score"] for row in rows], dtype=float) >= threshold
    tp = int((predicted * truth).sum())
    fp = int((predicted * (1 - truth)).sum())
    fn = int(((~predicted) * truth).sum())
    tn = int(((~predicted) * (1 - truth)).sum())
    sensitivity = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")
    precision = tp / (tp + fp) if tp + fp else float("nan")
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "n": len(rows),
            "sensitivity": sensitivity, "specificity": specificity, "precision": precision,
            "balanced_accuracy": (sensitivity + specificity) / 2,
            "accuracy": (tp + tn) / len(rows) if len(rows) else float("nan"),
            "f1": (2 * precision * sensitivity / (precision + sensitivity)
                   if math.isfinite(precision) and math.isfinite(sensitivity)
                   and (precision + sensitivity) > 0 else 0.0)}


operating_rows, selection_leaks, decision_records = [], [], []
for arm in ARMS_WITH_SCORES:
    rows = scored[arm]
    for rule in ["youden", "fixed_sensitivity"]:
        for fold in range(N_FOLDS):
            selection_rows = list(inner_scored.get(arm, {}).get(fold, []))
            test_rows = [row for row in rows if row["fold"] == fold]
            if len(selection_rows) < MIN_INNER_SELECTION_ROWS or len(test_rows) < 10:
                continue
            overlap = ({row["image_key"] for row in selection_rows}
                       & {row["image_key"] for row in test_rows})
            if overlap:
                selection_leaks.append((arm, rule, fold, len(overlap)))
            threshold, status = choose_threshold(selection_rows, rule)
            if threshold is None:
                continue
            applied = apply_threshold(test_rows, threshold)
            for test_row in test_rows:
                decision_records.append({
                    "arm": arm, "rule": rule, "fold": fold,
                    "threshold": threshold, "image_key": test_row["image_key"],
                    "patient": test_row["patient"], "truth": int(test_row["truth"]),
                    "score": float(test_row["score"]),
                    "predicted": int(float(test_row["score"]) >= threshold)})
            operating_rows.append({
                "arm": arm, "rule": rule, "fold": fold, "threshold": threshold,
                "n_selection_rows": len(selection_rows),
                "selection_score_coverage": float(np.mean(
                    [row["score_present"] for row in selection_rows])),
                "selection_status": status, "selected_on": "inner_validation",
                "applied_to": "held_out_fold", **applied})

operating_points = pd.DataFrame(operating_rows)
if len(operating_points):
    pooled_rows = []
    for (arm, rule), group in operating_points.groupby(["arm", "rule"]):
        totals = group[["tp", "fp", "tn", "fn", "n"]].sum()
        sensitivity = totals["tp"] / (totals["tp"] + totals["fn"])             if totals["tp"] + totals["fn"] else float("nan")
        specificity = totals["tn"] / (totals["tn"] + totals["fp"])             if totals["tn"] + totals["fp"] else float("nan")
        pooled_rows.append({
            "arm": arm, "rule": rule, "fold": "pooled", "threshold": None,
            "threshold_transfer_rule": (
                "fold-specific inner-validation thresholds applied within matching outer folds"),
            "transfer_threshold": round(float(group["threshold"].median()), 5),
            "threshold_mean_across_folds": round(float(group["threshold"].mean()), 5),
            "threshold_sd_across_folds": (round(float(group["threshold"].std(ddof=1)), 5)
                                           if len(group) > 1 else None),
            "selected_on": "inner_validation", "applied_to": "held_out_fold",
            **{key: int(totals[key]) for key in ["tp", "fp", "tn", "fn", "n"]},
            "sensitivity": sensitivity, "specificity": specificity,
            "balanced_accuracy": (sensitivity + specificity) / 2,
            "accuracy": ((totals["tp"] + totals["tn"]) / totals["n"]
                         if totals["n"] else float("nan"))})
    operating_points = pd.concat([operating_points, pd.DataFrame(pooled_rows)],
                                 ignore_index=True)
    operating_points.to_csv(NB18_DIR / "operating_points.csv", index=False)

    pooled = operating_points[operating_points["fold"] == "pooled"]
    print(pooled[["arm", "rule", "transfer_threshold", "threshold_sd_across_folds",
                  "sensitivity", "specificity", "balanced_accuracy", "tp", "fp", "tn",
                  "fn"]].to_string(index=False))
    print()
    print("Confusion counts are printed beside every rate on purpose: precision and F1 alone")
    print("hid the specificity collapse in the rejected version.")
    unstable = pooled[pd.to_numeric(
        pooled["threshold_sd_across_folds"], errors="coerce") > 0.15]
    if len(unstable):
        print()
        print(f"{len(unstable)} arm/rule combination(s) whose selected threshold moves by more")
        print("  than 0.15 across folds. A threshold that unstable is not a usable operating")
        print(f"  point: {unstable['arm'].tolist()[:5]}")
else:
    # Always replace a stale file from an earlier successful run with a readable schema.
    pd.DataFrame(columns=["arm", "rule", "fold", "threshold", "tp", "fp",
                          "tn", "fn", "n"]).to_csv(
        NB18_DIR / "operating_points.csv", index=False)
    print("No operating points could be selected.")

# Patient-level McNemar for the locked reasoner-vs-stacking pair. The stacking endpoint is
# the registered paired denominator; reference coverage must be a superset, never intersected
# down after the fact.
decision_frame = pd.DataFrame(decision_records)
pairing_issues = []
mcnemar_rows = []
expected_pair_keys = {row["image_key"] for row in scored.get(STACKING_ARM, [])}
for rule in ["youden", "fixed_sensitivity"]:
    if not expected_pair_keys:
        pairing_issues.append(
            f"{rule}: locked COVID stacking arm {STACKING_ARM!r} has no registered "
            "COVID endpoint rows; check NB 14 e7f_target.json and fusion predictions")
        continue
    if not len(decision_frame):
        pairing_issues.append(f"{rule}: no decision records were produced")
        continue
    subset = decision_frame[(decision_frame["rule"] == rule)
                            & decision_frame["arm"].isin([REFERENCE_ARM, STACKING_ARM])].copy()
    duplicates = subset.duplicated(["arm", "image_key"], keep=False)
    if duplicates.any():
        pairing_issues.append(
            f"{rule}: duplicate decision rows for {int(duplicates.sum())} arm/image entries")
        continue
    stack = subset[subset["arm"] == STACKING_ARM]
    reference = subset[subset["arm"] == REFERENCE_ARM]
    if stack.empty or reference.empty:
        pairing_issues.append(
            f"{rule}: decision rows are missing for "
            f"{'stacking' if stack.empty else 'reference'} arm")
        continue
    stack_keys = set(stack["image_key"].astype(str))
    reference_keys = set(reference["image_key"].astype(str))
    if stack_keys != expected_pair_keys:
        pairing_issues.append(
            f"{rule}: stacking decisions cover {len(stack_keys)} of "
            f"{len(expected_pair_keys)} locked COVID images")
        continue
    missing_reference = expected_pair_keys - reference_keys
    if missing_reference:
        pairing_issues.append(
            f"{rule}: reference arm misses {len(missing_reference)} stacking images")
        continue
    pair = pd.concat([stack, reference[reference["image_key"].isin(expected_pair_keys)]],
                     ignore_index=True)
    patient_rows = []
    for (arm, patient), group in pair.groupby(["arm", "patient"]):
        if group["truth"].nunique() != 1:
            raise RuntimeError(f"Patient {patient} has inconsistent PCR labels.")
        patient_rows.append({"arm": arm, "patient": patient,
                             "truth": int(group["truth"].iloc[0]),
                             "predicted": int(group["predicted"].mean() >= 0.5)})
    patient_frame = pd.DataFrame(
        patient_rows, columns=["arm", "patient", "truth", "predicted"])
    if patient_frame.empty:
        pairing_issues.append(f"{rule}: locked image set produced no patient rows")
        continue
    a = patient_frame[patient_frame["arm"] == STACKING_ARM].set_index("patient")
    b = patient_frame[patient_frame["arm"] == REFERENCE_ARM].set_index("patient")
    if set(a.index) != set(b.index):
        pairing_issues.append(
            f"{rule}: patient denominators differ after locking the image set "
            f"({len(a)} stacking vs {len(b)} reference)")
        continue
    shared = sorted(a.index.astype(str))
    if len(shared) < 20:
        pairing_issues.append(f"{rule}: only {len(shared)} paired patients")
        continue
    if not np.array_equal(a.loc[shared, "truth"].to_numpy(),
                          b.loc[shared, "truth"].to_numpy()):
        raise RuntimeError("Paired arms disagree on patient-level PCR truth.")
    truth_pair = a.loc[shared, "truth"].to_numpy(dtype=int)
    correct_a = a.loc[shared, "predicted"].to_numpy(dtype=int) == truth_pair
    correct_b = b.loc[shared, "predicted"].to_numpy(dtype=int) == truth_pair
    test = sd.mcnemar_test(correct_a, correct_b)
    patient_positions = bootstrap.patient_positions(shared)
    accuracy_difference = correct_a.astype(float) - correct_b.astype(float)
    accuracy_draws = bootstrap.resample_statistic(
        patient_positions, lambda weights: sd.weighted_mean(accuracy_difference, weights))
    accuracy_ci = sd.percentile_interval(accuracy_draws)
    p = sd.PValue(value=test["p"], test=test["method"], paired_unit="patient",
                  family="F4", family_size=2, adjusted=False,
                  effect=float(correct_a.mean() - correct_b.mean()),
                  effect_name="patient-level COVID accuracy difference", n=len(shared),
                  note="Holm adjustment is finalized with all F4/F5 rows in NB 19.")
    mcnemar_rows.append({"comparison": f"{STACKING_ARM} vs {REFERENCE_ARM}",
                         "arm_a": STACKING_ARM, "arm_b": REFERENCE_ARM,
                         "endpoint": f"COVID decision ({rule})", "family": "F4",
                         "n_images": len(expected_pair_keys), "n_patients": len(shared),
                         "delta": p.effect, "ci_low": accuracy_ci["ci_low"],
                         "ci_high": accuracy_ci["ci_high"], "test": p.test,
                         "paired_unit": p.paired_unit, "family_size": p.family_size,
                         "adjusted": False, "p_raw": p.value, "p_adjusted": None,
                         "b_a_right_b_wrong": test["b"], "c_b_right_a_wrong": test["c"],
                         "operating_point_source": "fold-specific inner validation",
                         "denominator_policy": "complete stacking COVID endpoint"})
comparison_columns = ["comparison", "arm_a", "arm_b", "endpoint", "family",
                      "n_images", "n_patients", "delta", "ci_low", "ci_high",
                      "test", "paired_unit", "family_size", "adjusted", "p_raw",
                      "p_adjusted", "b_a_right_b_wrong", "c_b_right_a_wrong",
                      "operating_point_source", "denominator_policy"]
pd.DataFrame(mcnemar_rows, columns=comparison_columns).to_csv(
    NB18_DIR / "operating_point_comparisons.csv", index=False)


## 5. Calibration

Brier score, expected calibration error, and a reliability curve per arm.

Discrimination and calibration are different properties and a paper needs both. An arm can rank
cases usefully while the probabilities it reports are meaningless — and given that Stage B found
detection AUROC barely above chance, the more interesting statement here may be that the
confidence values carry no information at all.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 12")

calibration_rows, reliability_curves = [], {}
for arm in ARMS_WITH_SCORES:
    rows = scored[arm]
    truth = np.array([r["truth"] for r in rows], dtype=float)
    score = np.array([r["score"] for r in rows], dtype=float)
    entry = {"arm": arm, "n": len(rows),
             "score_coverage": float(np.mean([r["score_present"] for r in rows])),
             "brier": float(cm.brier_score(truth.tolist(), score.tolist())),
             "ece": float(cm.expected_calibration_error(truth.tolist(), score.tolist(),
                                                        n_bins=N_CALIBRATION_BINS)),
             "mean_score": float(score.mean()), "prevalence": float(truth.mean()),
             "score_sd": float(score.std())}
    # A useful reference: the Brier score of always predicting the prevalence.
    entry["brier_of_prevalence_baseline"] = float(np.mean((truth - truth.mean()) ** 2))
    entry["beats_prevalence_baseline"] = bool(entry["brier"] < entry["brier_of_prevalence_baseline"])

    edges = np.linspace(0, 1, N_CALIBRATION_BINS + 1)
    points = []
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (score >= lower) & (score < upper if upper < 1 else score <= upper)
        if mask.sum() == 0:
            continue
        points.append({"bin_low": float(lower), "bin_high": float(upper),
                       "n": int(mask.sum()), "mean_score": float(score[mask].mean()),
                       "observed_rate": float(truth[mask].mean())})
    reliability_curves[arm] = points
    entry["n_populated_bins"] = len(points)
    calibration_rows.append(entry)

calibration_columns = ["arm", "n", "score_coverage", "brier", "ece",
                       "mean_score", "prevalence", "score_sd",
                       "brier_of_prevalence_baseline",
                       "beats_prevalence_baseline", "n_populated_bins"]
calibration = pd.DataFrame(calibration_rows, columns=calibration_columns)
if len(calibration):
    calibration = calibration.sort_values("brier")
calibration.to_csv(NB18_DIR / "calibration.csv", index=False)
curve_rows = [{"arm": arm, **point} for arm, points in reliability_curves.items()
              for point in points]
pd.DataFrame(curve_rows, columns=["arm", "bin_low", "bin_high", "n",
                                  "mean_score", "observed_rate"]).to_csv(
    NB18_DIR / "reliability_curves.csv", index=False)

print(calibration[["arm", "n", "score_coverage", "brier", "brier_of_prevalence_baseline",
                   "beats_prevalence_baseline", "ece", "score_sd"]].to_string(index=False))
worse = calibration[~calibration["beats_prevalence_baseline"]]["arm"].tolist()
if worse:
    print()
    print(f"{len(worse)} arm(s) score WORSE than predicting the cohort prevalence for every "
          "case:")
    print(f"  {worse[:6]}")
    print("  Their reported probabilities carry no usable information, whatever their AUROC.")
flat = calibration[calibration["score_sd"] < 0.05]["arm"].tolist()
if flat:
    print(f"{len(flat)} arm(s) emit an almost constant score (SD < 0.05): {flat[:6]}. "
          "A constant score cannot discriminate whatever the threshold.")

## 6. Figures 3 and 4

Vector output (PDF and SVG), no rasterised text, one shared style. Captions are generated
alongside so the numbers in them cannot drift from the figure.

Everything plotted here is read from the CSVs written above — the figure and the table are the
same numbers by construction rather than by care.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 14")

figure_paths = []
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                         "figure.dpi": 150, "savefig.bbox": "tight", "pdf.fonttype": 42,
                         "svg.fonttype": "none"})

    # ---- Figure 3: ROC and PR ------------------------------------------------------------
    if curve_data:
        figure, axes = plt.subplots(1, 2, figsize=(8.0, 3.6))
        for arm, data in curve_data.items():
            label = f"{arm} (AUROC {sd.format_number(data['auroc'], 'auroc')})"
            axes[0].plot(data["fpr"], data["tpr"], linewidth=1.4, label=label)
            axes[0].fill_between(data["fpr"], data["tpr_low"], data["tpr_high"], alpha=0.12,
                                 linewidth=0)
            axes[1].plot(data["recall"], data["precision"], linewidth=1.4, label=arm)
            axes[1].fill_between(data["recall"], data["precision_low"],
                                 data["precision_high"], alpha=0.12, linewidth=0)
        axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="chance")
        axes[0].set_xlabel("False positive rate (1 - specificity)")
        axes[0].set_ylabel("True positive rate (sensitivity)")
        axes[0].set_title("ROC, PCR-COVID")
        axes[0].legend(fontsize=6.5, loc="lower right", frameon=False)
        axes[1].set_xlabel("Recall")
        axes[1].set_ylabel("Precision")
        axes[1].set_title("Precision-recall")
        axes[1].legend(fontsize=6.5, loc="lower left", frameon=False)
        for path in [FIGURE_DIR / "fig3_roc_pr.pdf", FIGURE_DIR / "fig3_roc_pr.svg"]:
            figure.savefig(path)
            figure_paths.append(path)
        plt.close(figure)

    # ---- Figure 4: calibration ------------------------------------------------------------
    if reliability_curves:
        figure, axes = plt.subplots(1, 2, figsize=(8.0, 3.6))
        axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="perfect calibration")
        for arm in CURVE_ARMS:
            points = reliability_curves.get(arm) or []
            if not points:
                continue
            axes[0].plot([p["mean_score"] for p in points],
                         [p["observed_rate"] for p in points], marker="o", markersize=3,
                         linewidth=1.2, label=arm)
        axes[0].set_xlabel("Mean predicted probability")
        axes[0].set_ylabel("Observed positive rate")
        axes[0].set_title("Reliability diagram")
        axes[0].legend(fontsize=6.5, loc="upper left", frameon=False)

        top = calibration.head(min(10, len(calibration)))
        positions = np.arange(len(top))
        axes[1].barh(positions, top["brier"], height=0.6)
        for index, value in enumerate(top["brier_of_prevalence_baseline"]):
            axes[1].plot([value, value], [index - 0.35, index + 0.35], color="crimson",
                         linewidth=1.4)
        axes[1].set_yticks(positions)
        axes[1].set_yticklabels(top["arm"], fontsize=6.5)
        axes[1].invert_yaxis()
        axes[1].set_xlabel("Brier score (lower is better)")
        axes[1].set_title("Calibration vs the prevalence baseline (red)")
        for path in [FIGURE_DIR / "fig4_calibration.pdf", FIGURE_DIR / "fig4_calibration.svg"]:
            figure.savefig(path)
            figure_paths.append(path)
        plt.close(figure)
    print(f"Wrote {len(figure_paths)} figure file(s) to {FIGURE_DIR}")
except ImportError:
    print("matplotlib is unavailable; the CSVs above are complete and NB 21 can render from "
          "them. Install matplotlib before producing camera-ready figures.")

captions = []
if curve_data:
    best = max(curve_data.items(), key=lambda item: item[1]["auroc"])
    captions.append(
        "**Figure 3.** ROC and precision-recall curves for PCR-confirmed COVID-19 on the "
        f"pooled out-of-fold internal cohort. Shaded bands are patient-level bootstrap "
        f"intervals ({N_BAND_REPLICATES} replicates, drawn from the same indices as the "
        "confidence intervals in Table 2; the bands are a visual aid and the tabulated "
        "intervals use "
        f"{N_BOOTSTRAP} replicates). Highest AUROC: {best[0]} at "
        f"{sd.format_number(best[1]['auroc'], 'auroc')}.")
if len(calibration):
    worst_ece = calibration.sort_values("ece", ascending=False).iloc[0]
    captions.append(
        "**Figure 4.** Left: reliability diagram over "
        f"{N_CALIBRATION_BINS} equal-width probability bins. Right: Brier score per arm, with "
        "the red marker showing the Brier score obtained by predicting the cohort prevalence "
        "for every case; a bar extending past its marker indicates an arm whose probabilities "
        "are worse than that constant baseline. Largest expected calibration error: "
        f"{worst_ece['arm']} at {sd.format_number(worst_ece['ece'])}.")
(NB18_DIR / "captions.md").write_text("\n\n".join(captions) + "\n", encoding="utf-8")
print()
print("\n\n".join(captions))

## 7. Run configuration and gate

Blocking conditions:

1. **NB 13 and NB 17 passed their gates.** Partial or quarantined upstream artifacts cannot
   enter calibration.
2. **Endpoint denominators match NB 17, and no COVID-bearing row is duplicated.**
3. **No threshold was selected on the rows it is applied to.** This is the circularity that
   turns a threshold study into a self-report.
4. **The AUROCs here match the ones NB 17 locked.** If they do not, the two notebooks are
   reading different rows and one of the manuscript tables is wrong.
5. **The locked reasoner and stacking arms are COVID-score usable and have owner-produced
   inner-validation rows in all five folds.**
6. **All 2,000 shared patient-bootstrap replicates produce both ROC/PR bands, and both locked
   operating-point comparisons use the complete registered stacking denominator.**


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 18 before code cell 16")

failures, warnings = [], []

if denominator_issues:
    failures.append(
        f"COVID endpoint denominators disagree with NB 17 for {len(denominator_issues)} "
        f"arm(s), e.g. {denominator_issues[:3]}")
if inner_duplicate_issues:
    failures.append(
        f"Duplicate owner-produced inner-validation score rows found, e.g. "
        f"{inner_duplicate_issues[:3]}")

for arm, label in [(REFERENCE_ARM, "reference"), (STACKING_ARM, "stacking")]:
    if score_usable_by_arm.get(arm) is not True:
        failures.append(
            f"Locked {label} arm {arm!r} is not COVID-score usable in NB 17 "
            f"(score_usable={score_usable_by_arm.get(arm)!r}).")
    if arm not in ARMS_WITH_SCORES:
        failures.append(f"Locked {label} arm {arm!r} is absent from NB 18 score analyses.")

required_inner_missing = []
for arm in [REFERENCE_ARM, STACKING_ARM]:
    for fold in range(N_FOLDS):
        rows = inner_scored.get(arm, {}).get(fold, [])
        if len(rows) < MIN_INNER_SELECTION_ROWS or len({row["truth"] for row in rows}) < 2:
            required_inner_missing.append((arm, fold, len(rows)))
if required_inner_missing:
    failures.append(
        f"The locked reasoner/stacking comparison lacks adequate two-class, fold-specific "
        f"inner-validation scores for {required_inner_missing}. Pooled OOF scores from other "
        "fold models are not a valid substitute; produce these records in NB 14/NB 15.")
small_inner_sets = [(arm, fold, len(inner_scored.get(arm, {}).get(fold, [])))
                    for arm in [REFERENCE_ARM, STACKING_ARM]
                    for fold in range(N_FOLDS)
                    if MIN_INNER_SELECTION_ROWS <=
                    len(inner_scored.get(arm, {}).get(fold, [])) <
                    RECOMMENDED_INNER_SELECTION_ROWS]
if small_inner_sets:
    warnings.append(
        f"Thresholds are estimable but unstable because {len(small_inner_sets)} locked "
        f"arm/fold inner sets contain fewer than {RECOMMENDED_INNER_SELECTION_ROWS} "
        f"images: {small_inner_sets}. Report threshold variability and confusion counts.")

if selection_leaks:
    failures.append(
        f"{len(selection_leaks)} (arm, rule, fold) combination(s) selected their threshold on "
        f"rows they were then scored on, e.g. {selection_leaks[:3]}.")
else:
    print(f"THRESHOLD SELECTION VERIFIED: all {len(operating_points):,} operating-point rows "
          "were selected on inner validation and applied to held-out folds.")

if disagreements:
    failures.append(
        f"{len(disagreements)} arm(s) whose AUROC differs from NB 17 by more than 0.005, "
        f"e.g. {disagreements[0]}. Repair the row-set mismatch before reporting either.")
else:
    print(f"AUROC agreement with NB 17 verified for {len(ARMS_WITH_SCORES)} arms.")

if N_BAND_REPLICATES != N_BOOTSTRAP:
    failures.append(
        f"ROC/PR bands used {N_BAND_REPLICATES} replicates, not all {N_BOOTSTRAP} shared "
        "NB 17 patient-bootstrap replicates.")
if len(roc_pr):
    required_band_columns = ["tpr_ci_low", "tpr_ci_high",
                             "precision_ci_low", "precision_ci_high"]
    if any(column not in roc_pr or not np.isfinite(roc_pr[column]).any()
           for column in required_band_columns):
        failures.append("ROC and PR bootstrap bands were not both exported.")
else:
    failures.append("No ROC/PR curve rows were exported.")

if pairing_issues:
    failures.append(
        f"Locked operating-point pairing failed: {pairing_issues[:3]}")
if len(mcnemar_rows) != 2:
    failures.append(
        f"Expected two fixed-operating-point patient-level McNemar rows for the locked "
        f"reasoner/stacking pair; produced {len(mcnemar_rows)}.")

if len(operating_points):
    for arm in [REFERENCE_ARM, STACKING_ARM]:
        for rule in ["youden", "fixed_sensitivity"]:
            folds_present = set(pd.to_numeric(
                operating_points[(operating_points["arm"] == arm)
                                 & (operating_points["rule"] == rule)
                                 & (operating_points["fold"] != "pooled")]["fold"],
                errors="coerce").dropna().astype(int))
            if folds_present != set(range(N_FOLDS)):
                failures.append(
                    f"{arm}/{rule} has thresholds for folds {sorted(folds_present)}, expected "
                    f"{list(range(N_FOLDS))}.")

if no_score:
    warnings.append(
        f"{len(no_score)} arm(s) are quarantined by NB 17 or fall below "
        f"{MINIMUM_SCORE_COVERAGE:.0%} score coverage and are excluded from ROC/calibration: "
        f"{no_score[:5]}.")

if len(calibration):
    worse = calibration[~calibration["beats_prevalence_baseline"]]["arm"].tolist()
    if worse:
        warnings.append(
            f"{len(worse)} arm(s) have a Brier score worse than predicting prevalence: "
            f"{worse[:5]}.")
    flat = calibration[calibration["score_sd"] < 0.05]["arm"].tolist()
    if flat:
        warnings.append(f"{len(flat)} arm(s) emit a near-constant score: {flat[:5]}.")

if len(operating_points):
    pooled = operating_points[operating_points["fold"] == "pooled"]
    collapsed = pooled[pd.to_numeric(pooled["specificity"], errors="coerce") < 0.30]
    if len(collapsed):
        warnings.append(
            f"{len(collapsed)} arm/rule combination(s) have specificity below 0.30. "
            "Confusion counts in operating_points.csv must appear in the manuscript table.")
    unmet = [arm for arm in ARMS_WITH_SCORES
             if not len(pooled[(pooled["arm"] == arm)
                               & (pooled["rule"] == "fixed_sensitivity")])]
    if unmet:
        warnings.append(
            f"{len(unmet)} arm(s) lack a fixed-sensitivity pooled operating point: {unmet[:5]}.")

if not figure_paths:
    warnings.append("No figure files were written; NB 21 will have to render Figures 3 and 4 "
                    "from the CSVs.")

sd.write_json_atomic(NB18_DIR / "run_config.json", sd.provenance_stamp(
    "18_calibration_roc_and_thresholds.ipynb",
    {"n_arms_with_scores": len(ARMS_WITH_SCORES), "curve_arms": CURVE_ARMS,
     "reference_arm": REFERENCE_ARM, "stacking_arm": STACKING_ARM,
     "covid_stacking_arm": COVID_STACKING_ARM,
     "mrale_stacking_arm": MRALE_STACKING_ARM,
     "fixed_sensitivity_target": FIXED_SENSITIVITY,
     "minimum_inner_selection_rows": MIN_INNER_SELECTION_ROWS,
     "recommended_inner_selection_rows": RECOMMENDED_INNER_SELECTION_ROWS,
     "threshold_grid_points": len(GRID), "calibration_bins": N_CALIBRATION_BINS,
     "band_replicates": N_BAND_REPLICATES,
     "bootstrap_fingerprint": bootstrap.fingerprint,
     "bootstrap_mode": "read_only; all shared NB17 replicates; mismatch is blocking",
     "upstream_gates": UPSTREAM_GATES,
     "endpoint_usability_path": str(usability_path),
     "endpoint_denominator_issues": denominator_issues,
     "inner_score_sources": inner_score_status,
     "mcnemar_rows": len(mcnemar_rows), "pairing_issues": pairing_issues,
     "figures": [str(path) for path in figure_paths],
     "threshold_policy": ("selected from owner-produced continuous scores restricted by "
                          "NB 13 fold-relative membership, then applied to the matching "
                          "held-out fold"),
     "invalid_score_policy": "retain as 0.5; never selectively drop"}))


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
sd.write_json_atomic(NB18_DIR / "gate_nb18.json",
                     {"passed": not failures, "failures": failures, "warnings": warnings,
                      "upstream_gates": UPSTREAM_GATES,
                      "bootstrap_fingerprint": bootstrap.fingerprint})
if failures:
    detail = "\n".join(f"  [{index + 1}] {message}"
                       for index, message in enumerate(failures))
    raise AssertionError(f"NB 18 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 18 gate: PASSED")
